## Importacion de librerias

In [1]:
import tensorflow as tf
from tensorflow import keras
import sklearn
from keras import layers

## Pruebas individuales

In [2]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
magic_gamma_telescope = fetch_ucirepo(id=159) 
  
# data (as pandas dataframes) 
X = magic_gamma_telescope.data.features 
y = magic_gamma_telescope.data.targets 
  
# metadata 
print(magic_gamma_telescope.metadata) 
  
# variable information 
print(magic_gamma_telescope.variables) 

{'uci_id': 159, 'name': 'MAGIC Gamma Telescope', 'repository_url': 'https://archive.ics.uci.edu/dataset/159/magic+gamma+telescope', 'data_url': 'https://archive.ics.uci.edu/static/public/159/data.csv', 'abstract': 'Data are MC generated to simulate registration of high energy gamma particles in an atmospheric Cherenkov telescope', 'area': 'Physics and Chemistry', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 19020, 'num_features': 10, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2004, 'last_updated': 'Tue Dec 19 2023', 'dataset_doi': '10.24432/C52C8B', 'creators': ['R. Bock'], 'intro_paper': None, 'additional_info': {'summary': "The data are MC generated (see below) to simulate registration of high energy gamma particles in a ground-based atmospheric Cherenkov gamma telescope using the imaging technique. Cherenkov gamm

In [4]:
# Capa personalizada que implementa una combinación de polinomios de Legendre
class PolynomialDense(tf.keras.layers.Layer):
    def __init__(self, units, degree=2, use_bias=True, **kwargs):
        super(PolynomialDense, self).__init__(**kwargs)
        self.units = units
        self.degree = degree
        self.use_bias = use_bias

    def build(self, input_shape):
        input_dim = input_shape[-1]

        self.kernels = []
        for d in range(1, self.degree + 1):
            w = self.add_weight(
                shape=(input_dim, self.units),
                initializer='glorot_uniform',
                trainable=True,
                name=f'kernel_degree_{d}'
            )
            self.kernels.append(w)

        if self.use_bias:
            self.bias = self.add_weight(
                shape=(self.units,),
                initializer='zeros',
                trainable=True,
                name='bias'
            )

    def call(self, inputs):

        #Asumimos que los inputs ya están normalizados en [-1, 1]
        x = inputs

        # --- 2. Generamos la base de Legendre
        # P0(x)
        P0 = tf.ones_like(x)

        if self.degree == 0:
            features = [P0]
        else:
            # P1(x)
            P1 = x
            features = [P1]

            if self.degree > 1:
                Pnm2 = P0
                Pnm1 = P1

                for n in range(2, self.degree + 1):
                    Pn = ((2.0 * n - 1.0) * x * Pnm1 - (n - 1.0) * Pnm2) / n
                    features.append(Pn)

                    Pnm2 = Pnm1
                    Pnm1 = Pn

        # --- 3. Combinación lineal con los kernels
        output = 0.0

        for phi, W in zip(features, self.kernels):
            output += tf.matmul(phi, W)

        if self.use_bias:
            output += self.bias

        return output


In [5]:
class PolynomialDense2(tf.keras.layers.Layer):
    def __init__(self, units, degree=2, use_bias=True, **kwargs):
        super(PolynomialDense2, self).__init__(**kwargs)
        self.units = units
        self.degree = degree
        self.use_bias = use_bias

    def build(self, input_shape):
        input_dim = input_shape[-1]
        # El kernel debe cubrir (grado + 1) si incluyes P0, o (grado) si empiezas en P1
        # Aquí usaremos desde P1 hasta P_degree
        self.kernel = self.add_weight(
            shape=(input_dim * self.degree, self.units),
            initializer=tf.keras.initializers.GlorotUniform(),
            trainable=True,
            name="kernel"
        )

        if self.use_bias:
            self.bias = self.add_weight(
                shape=(self.units,),
                initializer="zeros",
                trainable=True,
                name="bias"
            )

    def call(self, inputs):
        # Aseguramos que los inputs sean float32
        x = tf.cast(inputs, self.compute_dtype)
        
        # P0 = 1, P1 = x
        p_nm2 = tf.ones_like(x)
        p_nm1 = x
        
        features = [p_nm1] # Empezamos con grado 1

        for n in range(2, self.degree + 1):
            n_float = tf.cast(n, self.compute_dtype)
            # Fórmula de recurrencia de Legendre
            p_n = ((2.0 * n_float - 1.0) * x * p_nm1 - (n_float - 1.0) * p_nm2) / n_float
            features.append(p_n)
            
            p_nm2 = p_nm1
            p_nm1 = p_n

        # Concatenamos: (batch, input_dim * degree)
        # Esto genera: [x_poly1, x_poly2, ..., x_poly_degree]
        phi = tf.concat(features, axis=-1)

        output = tf.matmul(phi, self.kernel)

        if self.use_bias:
            output = tf.nn.bias_add(output, self.bias)

        return output

In [8]:
class PolynomialDenseV3(tf.keras.layers.Layer):
    def __init__(self, units, degree=2, use_bias=True, **kwargs):
        super(PolynomialDenseV3, self).__init__(**kwargs)
        self.units = units
        self.degree = degree
        self.use_bias = use_bias

    def build(self, input_shape):
        input_dim = input_shape[-1]
        
        # Inicialización optimizada: escala pequeña para estabilidad inicial
        # Esto evita que los términos de grado alto dominen al principio.
        stddev = 1.0 / (tf.sqrt(tf.cast(input_dim, tf.float32)) * self.degree)
        
        self.kernel = self.add_weight(
            shape=(input_dim * self.degree, self.units),
            initializer=tf.keras.initializers.RandomNormal(stddev=stddev),
            trainable=True,
            name="kernel"
        )

        if self.use_bias:
            self.bias = self.add_weight(
                shape=(self.units,),
                initializer="zeros",
                trainable=True,
                name="bias"
            )

    def call(self, inputs):
        x = tf.cast(inputs, self.compute_dtype)
        
        # P0 = 1, P1 = x
        p_nm2 = tf.ones_like(x)
        p_nm1 = x
        
        # Guardamos los términos en una lista para concatenar al final
        # Usamos una estructura que mantenga el orden (batch, degree, input_dim)
        features = [p_nm1]

        for n in range(2, self.degree + 1):
            n_float = tf.cast(n, self.compute_dtype)
            p_n = ((2.0 * n_float - 1.0) * x * p_nm1 - (n_float - 1.0) * p_nm2) / n_float
            features.append(p_n)
            
            p_nm2 = p_nm1
            p_nm1 = p_n

        # --- MEJORA: Concatenación inteligente ---
        # phi shape: (batch, input_dim * degree)
        phi = tf.concat(features, axis=-1)

        # Transformación lineal principal
        output = tf.matmul(phi, self.kernel)

        if self.use_bias:
            output = tf.nn.bias_add(output, self.bias)

        return output

    # --- MEJORA: Hacer la capa serializable ---
    def get_config(self):
        config = super(PolynomialDenseV3, self).get_config()
        config.update({
            "units": self.units,
            "degree": self.degree,
            "use_bias": self.use_bias,
        })
        return config

In [9]:
#Dividimos el dataset en entrenamiento y prueba
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, test_size=0.3, random_state=1)

In [10]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((13314, 10), (5706, 10), (13314, 1), (5706, 1))

In [11]:
#Normalizamos los datos
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(-1, 1))
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
#Cambiamos las etiquetas a 0 y 1
y_train = (y_train == 'g').astype(int)
y_test = (y_test == 'g').astype(int)

### Entrenamiento de los modelos

In [13]:
input_dim = X_train.shape[1]

#Modelo lineal
inputLinear = keras.Input(shape=(input_dim,))
x1 = layers.Dense(32, activation='relu')(inputLinear)
x2 = layers.Dense(16, activation='relu')(x1)
outputLinear = layers.Dense(3, activation='softmax')(x2)

#MODELO DE RED POLINÓMICA DE LEGENDRE

#Modelo polinómico grado 2

inputPoli = keras.Input(shape=(input_dim,))
x = PolynomialDense2(32, degree=2)(inputPoli)
x = layers.Activation('swish')(x)
x = layers.Dense(16, activation='swish')(x)
outputPoli = layers.Dense(3, activation='softmax')(x)

#Modelo polinómico grado 3
inputPoli3 = keras.Input(shape=(input_dim,))
x3 = PolynomialDense2(32, degree=3)(inputPoli3)
x3 = layers.Activation('swish')(x3)
x3 = layers.Dense(16, activation='swish')(x3)
outputPoli3 = layers.Dense(3, activation='softmax')(x3)

#Modelo polinómico grado 4
inputPoli4 = keras.Input(shape=(input_dim,))
x4 = PolynomialDense2(32, degree=4)(inputPoli4)
x4 = layers.Activation('swish')(x4)
x4 = layers.Dense(16, activation='swish')(x4)
outputPoli4 = layers.Dense(3, activation='softmax')(x4)


In [14]:
#Implementamos el polinomioV3 grados 2,3,4

inputPoliV3 = keras.Input(shape=(input_dim,))
xV3 = PolynomialDenseV3(32, degree=2)(inputPoliV3)
xV3 = layers.BatchNormalization()(xV3)  # Normalización para estabilizar el entrenamiento
xV3 = layers.Activation('swish')(xV3)
xV3 = layers.Dense(16, activation='swish')(xV3)
outputPoliV3 = layers.Dense(3, activation='softmax')(xV3)

inputPoliV3G3 = keras.Input(shape=(input_dim,))
xV3G3 = PolynomialDenseV3(32, degree=3)(inputPoliV3G3)
xV3G3 = layers.BatchNormalization()(xV3G3)  # Normalización para estabilizar el entrenamiento
xV3G3 = layers.Activation('swish')(xV3G3)
xV3G3 = layers.Dense(16, activation='swish')(xV3G3)
outputPoliV3G3 = layers.Dense(3, activation='softmax')(xV3G3)

inputPoliV3G4 = keras.Input(shape=(input_dim,))
xV3G4 = PolynomialDenseV3(32, degree=4)(inputPoliV3G4)
xV3G4 = layers.BatchNormalization()(xV3G4)  # Normalización para estabilizar el entrenamiento
xV3G4 = layers.Activation('swish')(xV3G4)
xV3G4 = layers.Dense(16, activation='swish')(xV3G4)
outputPoliV3G4 = layers.Dense(3, activation='softmax')(xV3G4)


In [15]:
#Guardamos ambos modelos
modelLinear  = keras.Model(inputs=inputLinear, outputs=outputLinear)

modelPoli = keras.Model(inputs=inputPoli, outputs=outputPoli) 
modelPoli3 = keras.Model(inputs=inputPoli3, outputs=outputPoli3)
modelPoli4 = keras.Model(inputs=inputPoli4, outputs=outputPoli4)


In [16]:
modelV3 = keras.Model(inputs=inputPoliV3, outputs=outputPoliV3)
modelV3G3 = keras.Model(inputs=inputPoliV3G3, outputs=outputPoliV3G3)
modelV3G4 = keras.Model(inputs=inputPoliV3G4, outputs=outputPoliV3G4)

In [91]:
modelV3.summary()

Model: "functional_36"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_32 (InputLayer)     │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ polynomial_dense_v3_12          │ (None, 32)             │           672 │
│ (PolynomialDenseV3)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_27 (Activation)      │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_69 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_70 (Dense)                │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,379 (5.39 KB)

 Trainable params: 1,315 (5.14 KB)

 Non-trainable params: 64 (256.00 B)

In [ ]:
from tensorflow.keras import optimizers
#Compilamos ambos modelos
modelLinear.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

modelPoli.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelPoli3.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelPoli4.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])


In [93]:
modelV3.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelV3G3.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelV3G4.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])   


In [94]:
epochs = 100

In [95]:
#Añadimos Early Stopping
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
#Entrenamos el modelo
historyLinear = modelLinear.fit(X_train, y_train, epochs=epochs, batch_size=16, validation_data=(X_test, y_test), callbacks=[early_stopping])

historyPoli = modelPoli.fit(X_train, y_train, epochs=epochs, batch_size=16, validation_data=(X_test, y_test), callbacks=[early_stopping])
historyPoli3 = modelPoli3.fit(X_train, y_train, epochs=epochs, batch_size=16, validation_data=(X_test, y_test), callbacks=[early_stopping])
historyPoli4 = modelPoli4.fit(X_train, y_train, epochs=epochs, batch_size=16, validation_data=(X_test, y_test), callbacks=[early_stopping])

#Entrenamos el modeloV3
historyV3 = modelV3.fit(X_train, y_train, epochs=epochs, batch_size=16, validation_data=(X_test, y_test), callbacks=[early_stopping])
historyV3G3 = modelV3G3.fit(X_train, y_train, epochs=epochs, batch_size=16, validation_data=(X_test, y_test), callbacks=[early_stopping])
historyV3G4 = modelV3G4.fit(X_train, y_train, epochs=epochs, batch_size=16, validation_data=(X_test, y_test), callbacks=[early_stopping])

Epoch 1/100
833/833 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.7791 - loss: 0.5025 - val_accuracy: 0.7990 - val_loss: 0.4308
Epoch 2/100
833/833 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8179 - loss: 0.4098 - val_accuracy: 0.8319 - val_loss: 0.3942
Epoch 3/100
833/833 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8349 - loss: 0.3842 - val_accuracy: 0.8323 - val_loss: 0.3824
Epoch 4/100
833/833 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8408 - loss: 0.3711 - val_accuracy: 0.8363 - val_loss: 0.3745
Epoch 5/100
833/833 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8504 - loss: 0.3632 - val_accuracy: 0.8577 - val_loss: 0.3569
Epoch 6/100
833/833 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8550 - loss: 0.3565 - val_accuracy: 0.8456 - val_loss: 0.3641
Epoch 7/100
833/833 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8545 - loss: 0.3547 - val_accuracy: 0.8580 - val_loss: 0.3505
Epoch 8/100
833/833 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8545 - loss: 0.3508 - val_accu

In [ ]:
#Comparamos los resultados de ambos modelos
lossLinear, accLinear = modelLinear.evaluate(X_test, y_test, verbose=0)

lossPoli, accPoli = modelPoli.evaluate(X_test, y_test, verbose=0)
lossPoli3, accPoli3 = modelPoli3.evaluate(X_test, y_test, verbose=0)
lossPoli4, accPoli4 = modelPoli4.evaluate(X_test, y_test, verbose=0)



lossV3, accV3 = modelV3.evaluate(X_test, y_test, verbose=0)
lossV3G3, accV3G3 = modelV3G3.evaluate(X_test, y_test, verbose=0)
lossV3G4, accV3G4 = modelV3G4.evaluate(X_test, y_test, verbose=0)

print(f'Modelo Lineal - Loss: {lossLinear:.4f}, Accuracy: {accLinear:.4f}')
print(f'Modelo Polinómico Grado 2 - Loss: {lossPoli:.4f}, Accuracy: {accPoli:.4f}')
print(f'Modelo Polinómico Grado 3 - Loss: {lossPoli3:.4f}, Accuracy: {accPoli3:.4f}')
print(f'Modelo Polinómico Grado 4 - Loss: {lossPoli4:.4f}, Accuracy: {accPoli4:.4f}')



print(f'Modelo V3 Grado 2 - Loss: {lossV3:.4f}, Accuracy: {accV3:.4f}')
print(f'Modelo V3 Grado 3 - Loss: {lossV3G3:.4f}, Accuracy: {accV3G3:.4f}')
print(f'Modelo V3 Grado 4 - Loss: {lossV3G4:.4f}, Accuracy: {accV3G4:.4f}')

Modelo Polinómico V2 Grado 2 - Loss: 0.3804, Accuracy: 0.8337
Modelo Polinómico V2 Grado 3 - Loss: 0.3661, Accuracy: 0.8465
Modelo Polinómico V2 Grado 4 - Loss: 0.3579, Accuracy: 0.8502
Modelo V3 Grado 2 - Loss: 0.3508, Accuracy: 0.8568
Modelo V3 Grado 3 - Loss: 0.3017, Accuracy: 0.8768
Modelo V3 Grado 4 - Loss: 0.3407, Accuracy: 0.8577


# Pruba con Cross-Validation

In [23]:
from tensorflow.keras import optimizers

In [24]:
#creamos una funcion para crear un modelo lineal y un modelo polinómico de Legendre

def build_linear_model():
    inputLinear = keras.Input(shape=(input_dim,))
    x1 = layers.Dense(32, activation='relu')(inputLinear)
    x2 = layers.Dense(16, activation='relu')(x1)
    outputLinear = layers.Dense(3, activation='softmax')(x2)
    modelLinear = keras.Model(inputs=inputLinear, outputs=outputLinear)
    modelLinear.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    return modelLinear

def build_poly_model(degree):
    inputPoli = keras.Input(shape=(input_dim,))
    x = PolynomialDense2(32, degree=degree)(inputPoli)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    x = layers.Dense(16, activation='swish')(x)
    outputPoli = layers.Dense(3, activation='softmax')(x)
    modelPoli = keras.Model(inputs=inputPoli, outputs=outputPoli)
    modelPoli.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return modelPoli

## Cross Validation.

In [25]:
import numpy as np
from tqdm import tqdm #Permite mostrar una barra de progreso durante el entrenamiento

In [26]:
X = magic_gamma_telescope.data.features 
y = magic_gamma_telescope.data.targets 

In [27]:
epochs = 20

In [28]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)

scoreLinear = []
scorePoli = []
scorePoli3 = []
scorePoli4 = []

X = X.to_numpy()
y = y.to_numpy()



for train_index, test_index in tqdm(skf.split(X, y), total=10):

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Normalización
    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # etiquetas binarias
    y_train = (y_train == 'g').astype(int)
    y_test = (y_test == 'g').astype(int)

    #CREAR MODELOS NUEVOS EN CADA FOLD
    modelLinear = build_linear_model()
    modelPoli = build_poly_model(2)
    modelPoli3 = build_poly_model(3)
    modelPoli4 = build_poly_model(4)

    # Early stopping nuevo
    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    # Entrenamiento
    modelLinear.fit(X_train, y_train, epochs=epochs, batch_size=16,
                    validation_split=0.1,
                    callbacks=[early_stopping],
                    verbose=0)

    modelPoli.fit(X_train, y_train, epochs=epochs, batch_size=16,
                  validation_split=0.1,
                  callbacks=[early_stopping],
                  verbose=0)

    modelPoli3.fit(X_train, y_train, epochs=epochs, batch_size=16,
                   validation_split=0.1,
                   callbacks=[early_stopping],
                   verbose=0)

    modelPoli4.fit(X_train, y_train, epochs=epochs, batch_size=16,
                   validation_split=0.1,
                   callbacks=[early_stopping],
                   verbose=0)

    # Evaluación
    _, accLinear = modelLinear.evaluate(X_test, y_test, verbose=0)
    _, accPoli = modelPoli.evaluate(X_test, y_test, verbose=0)
    _, accPoli3 = modelPoli3.evaluate(X_test, y_test, verbose=0)
    _, accPoli4 = modelPoli4.evaluate(X_test, y_test, verbose=0)

    scoreLinear.append(accLinear)
    scorePoli.append(accPoli)
    scorePoli3.append(accPoli3)
    scorePoli4.append(accPoli4)

100%|██████████| 10/10 [12:33<00:00, 75.37s/it]


In [29]:
#imprimimos los resultados
print(f'Modelo Lineal - Accuracy: {np.mean(scoreLinear):.4f}' ' ± ' + f'{np.std(scoreLinear):.4f}')
print(f'Modelo Polinómico - Accuracy: {np.mean(scorePoli):.4f}' ' ± ' + f'{np.std(scorePoli):.4f}')
print(f'Modelo Polinómico 3 - Accuracy: {np.mean(scorePoli3):.4f}' ' ± ' + f'{np.std(scorePoli3):.4f}')
print(f'Modelo Polinómico 4 - Accuracy: {np.mean(scorePoli4):.4f}' ' ± ' + f'{np.std(scorePoli4):.4f}')

Modelo Lineal - Accuracy: 0.8251 ± 0.0081
Modelo Polinómico - Accuracy: 0.8621 ± 0.0046
Modelo Polinómico 3 - Accuracy: 0.8142 ± 0.0299
Modelo Polinómico 4 - Accuracy: 0.8050 ± 0.0237


### Con Dataset de Iris

In [63]:
from sklearn.datasets import load_iris

In [65]:
iris = load_iris()

X_iris = iris.data
y_iris = iris.target
X_train_iris, X_test_iris, y_train_iris, y_test_iris = sklearn.model_selection.train_test_split(X_iris, y_iris, test_size=0.3, random_state=1)

In [67]:
import tensorflow as tf
from tensorflow.keras import backend as K

# --- Ajuste en las funciones de construcción ---

# 2. Construcción del modelo lineal simple
def build_simple_linear_model():
    model = keras.Sequential([
        keras.Input(shape=(input_dim,)),
        layers.Dense(16, activation='relu'),
        layers.Dense(8, activation='relu'),
        layers.Dense(3, activation='softmax') # 3 clases de flores
    ])
    
    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def build_poly_model_iris(degree):
    inputPoli = keras.Input(shape=(4,))
    x = PolynomialDenseV3(16, degree=degree)(inputPoli) 
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    outputPoli = layers.Dense(3, activation='softmax')(x)
    
    modelPoli = keras.Model(inputs=inputPoli, outputs=outputPoli)
    modelPoli.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3), 
        loss='sparse_categorical_crossentropy', 
        metrics=['accuracy']
    )
    return modelPoli

# --- Bucle de ejecución corregido ---

for train_index, test_index in tqdm(skf.split(X_iris, y_iris), total=10):
    # 1. Limpiar la sesión al inicio de cada FOLD para resetear el optimizador
    K.clear_session()
    
    X_train, X_test = X_iris[train_index], X_iris[test_index]
    y_train, y_test = y_iris[train_index], y_iris[test_index]

    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # 2. Re-instanciar el EarlyStopping en cada fold
    early_stopping = keras.callbacks.EarlyStopping(
        monitor="loss", 
        patience=50, 
        restore_best_weights=True
    )

    models_fold = {
        "linear": build_simple_linear_model(),
        "poly2": build_poly_model_iris(2),
        "poly3": build_poly_model_iris(3),
        "poly4": build_poly_model_iris(4)
    }

    for name, model in models_fold.items():
        model.fit(
            X_train, y_train, 
            epochs=200, # Bajamos de 1000 a 200, con LR 1e-3 sobra
            batch_size=16, # Subimos un poco el batch
            verbose=0,
            callbacks=[early_stopping]
        )
        _, acc = model.evaluate(X_test, y_test, verbose=0)
        results[name].append(acc)

# Mostrar resultados finales
for name, scores in results.items():
    print(f"Modelo {name}: Accuracy promedio = {np.mean(scores):.4f}")

100%|██████████| 10/10 [04:24<00:00, 26.47s/it]

Modelo linear: Accuracy promedio = 0.7625
Modelo poly2: Accuracy promedio = 0.4667
Modelo poly3: Accuracy promedio = 0.4521
Modelo poly4: Accuracy promedio = 0.5667


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# 1. Preparación de los datos
iris = load_iris()
X, y = iris.data, iris.target

# Dividimos en 80% entrenamiento y 20% prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalización (Importante para redes neuronales)
scaler = MinMaxScaler(feature_range=(-1, 1))
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

input_dim = X.shape[1] # Esto será 4 para Iris



model = build_simple_linear_model()

# 3. Entrenamiento
print("Entrenando el modelo lineal...")
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=8,
    validation_split=0.1, # Usamos un trozo del train para validar durante el proceso
    verbose=0 # Silencioso para no llenar la consola
)

# 4. Evaluación final
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print("-" * 30)
print(f"Resultado en el conjunto de TEST:")
print(f"Loss: {loss:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print("-" * 30)

# Ejemplo de predicción simple
sample = X_test[:1]
prediction = model.predict(sample, verbose=0)
print(f"Predicción para la primera muestra: {np.argmax(prediction)} (Real: {y_test[0]})")

Entrenando el modelo lineal...
------------------------------
Resultado en el conjunto de TEST:
Loss: 0.0394
Accuracy: 1.0000
------------------------------
Predicción para la primera muestra: 1 (Real: 1)
